In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
CATALOG = "workspace"
SCHEMA = "retail_sales"

BASE_SCHEMA = f"{CATALOG}.{SCHEMA}"

print("Working schema:", BASE_SCHEMA)

In [0]:
bronze_sales = spark.table(
    f"{BASE_SCHEMA}.bronze_store_sales"
)

bronze_customers = spark.table(
    f"{BASE_SCHEMA}.bronze_customers"
)

bronze_items = spark.table(
    f"{BASE_SCHEMA}.bronze_items"
)

bronze_stores = spark.table(
    f"{BASE_SCHEMA}.bronze_stores"
)

bronze_dates = spark.table(
    f"{BASE_SCHEMA}.bronze_dates"
)

print("ALL Bronze loaded successfully")

In [0]:
bronze_tables = {
    "Sales": bronze_sales,
    "Customers": bronze_customers,
    "Items": bronze_items,
    "Stores": bronze_stores,
    "Dates": bronze_dates
}

for name, df in bronze_tables.items():
    print(f"{name}: {df.count()} rows")

In [0]:
bronze_sales.printSchema()

In [0]:
columns_to_check = [
    "ss_item_sk",
    "ss_ticket_number",
    "ss_customer_sk",
    "ss_store_sk",
    "ss_quantity",
    "ss_sales_price",
    "ss_net_paid"
]

null_expressions = [
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in columns_to_check
]

null_report = bronze_sales.select(
    null_expressions
)

display(null_report)

In [0]:
sales_prepared = bronze_sales.select(
    F.col("ss_sold_date_sk").alias("date_key"),
    F.col("ss_item_sk").alias("item_key"),
    F.col("ss_customer_sk").alias("customer_key"),
    F.col("ss_store_sk").alias("store_key"),
    F.col("ss_ticket_number").alias("ticket_number"),
    F.col("ss_quantity").cast("int").alias("quantity"),
    F.col("ss_sales_price").cast("decimal(18,2)").alias("sales_price"),
    F.col("ss_net_paid").cast("decimal(18,2)").alias("net_revenue"),
    F.col("ingestion_run_id"),
    F.col("ingested_at")
)

display(sales_prepared.limit(10))

In [0]:
sales_validated = sales_prepared.withColumn(
    "validation_error",
    F.when(
        F.col("ticket_number").isNull() |
        F.col("item_key").isNull(),
        F.lit("Missing_Transaction_Key")
    )
    .when(
        F.col("quantity").isNull() |
        (F.col("quantity") <= 0),
        F.lit("INVALID_QUANTITY")
    )
    .when(
        F.col("net_revenue").isNull() |
        (F.col("net_revenue") < 0),
        F.lit("INVALID_NET_REVENUE")
    )
    .when(
        F.col("sales_price").isNull() |
        (F.col("sales_price") < 0),
        F.lit("INVALID_SALES_PRICE")
    )
    .otherwise(F.lit(None).cast("string"))
)

display(
    sales_validated.groupBy("validation_error")
    .count()
)

In [0]:
valid_sales = (
    sales_validated
    .filter(F.col("validation_error").isNull())
    .drop("validation_error")
)

rejected_sales = (
    sales_validated
    .filter(F.col("validation_error").isNotNull())
)

print("Valid rows:", valid_sales.count())
print("Rejected rows:", rejected_sales.count())

In [0]:
duplicate_keys = (
    valid_sales
    .groupBy("ticket_number", "item_key")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_keys)

print(
    "Duplicate key groups:",
    duplicate_keys.count()
)

In [0]:
business_columns = [
    "date_key",
    "item_key",
    "customer_key",
    "store_key",
    "ticket_number",
    "quantity",
    "sales_price",
    "net_revenue"
]

sales_no_exact_duplicates = (
    valid_sales.dropDuplicates(business_columns)
)

conflicting_keys = (
    sales_no_exact_duplicates
    .groupBy("ticket_number", "item_key")
    .count()
    .filter(F.col("count") > 1)
    .select("ticket_number", "item_key")
)

conflicting_sales = (
    sales_no_exact_duplicates
    .join(
        conflicting_keys,
        ["ticket_number", "item_key"],
        "inner"
    )
    .withColumn(
        "validation_error",
        F.lit("CONFLICTING_TRANSACTION_KEY")
    )
)

silver_sales = (
    sales_no_exact_duplicates
    .join(
        conflicting_keys,
        ["ticket_number", "item_key"],
        "left_anti"
    )
)

print("Silver sales rows:", silver_sales.count())
print("Conflicting rows:", conflicting_sales.count())

In [0]:
all_rejected_sales = rejected_sales.unionByName(
    conflicting_sales
)

display(
    all_rejected_sales.groupBy("validation_error")
    .count()
)

In [0]:
def clean_dimension(
    df,
    columns,
    key_column
):
    
    prepared = df.select(
        [F.col(source).alias(target) for source, target in columns.items()]
    )

    prepared = (
        prepared
        .filter(F.col(key_column).isNotNull())
        .dropDuplicates()
    )

    duplicate_keys= (
        prepared
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .select(key_column)
    )

    cleaned = prepared.join(
        duplicate_keys,
        key_column,
        "left_anti"
    )

    rejected = prepared.join(
        duplicate_keys,
        key_column,
        "inner"
    )

    return cleaned, rejected





In [0]:
silver_customers, rejected_customers = clean_dimension(
    bronze_customers,
    {
        "c_customer_sk": "customer_key",
        "c_customer_id": "customer_id",
        "c_first_name": "first_name",
        "c_last_name": "last_name",
        "c_birth_year": "birth_year"
    },
    "customer_key"
)

display(silver_customers.limit(10))

In [0]:
silver_items, rejected_items = clean_dimension(
    bronze_items,
    {
        "i_item_sk": "item_key",
        "i_item_id": "item_id",
        "i_product_name": "product_name",
        "i_category": "category",
        "i_class": "product_class",
        "i_current_price": "current_price"
    },
    "item_key"
)

display(silver_items.limit(10))

In [0]:
silver_stores, rejected_stores = clean_dimension(
    bronze_stores,
    {
        "s_store_sk": "store_key",
        "s_store_id": "store_id",
        "s_store_name": "store_name",
        "s_city": "city",
        "s_state": "state"
    },
    "store_key"
)

display(silver_stores.limit(10))

In [0]:
silver_dates, rejected_dates = clean_dimension(
    bronze_dates,
    {
        "d_date_sk": "date_key",
        "d_date": "calendar_date",
        "d_year": "year",
        "d_moy": "month",
        "d_qoy": "quarter"
    },
    "date_key"
)

silver_dates = silver_dates.withColumn(
    "calendar_date",
    F.to_date("calendar_date")
)

display(silver_dates.limit(10))

In [0]:
def check_relationship(
    fact_df,
    dimension_df,
    key_column
):
    
    populated_keys = fact_df.filter(
        F.col(key_column).isNotNull()
    )

    unmatched = populated_keys.join(
        dimension_df.select(key_column).distinct(),
        key_column,
        "left_anti"
    )

    return unmatched.count()

In [0]:
unmatched_items = check_relationship(
    silver_sales, silver_items, "item_key"
)

unmatched_customers = check_relationship(
    silver_sales, silver_customers, "customer_key"
)

unmatched_stores = check_relationship(
    silver_sales, silver_stores, "store_key"
)

unmatched_dates = check_relationship(
    silver_sales, silver_dates, "date_key"
)

print("Unmatched products:", unmatched_items)
print("Unmatched customers:", unmatched_customers)
print("Unmatched stores:", unmatched_stores)
print("Unmatched dates:", unmatched_dates)

print(
    "Sales with null customer keys:",
    silver_sales.filter(
        F.col("customer_key").isNull()
    ).count()
)

In [0]:
def save_silver_table(df, table_name):
    full_name = f"{BASE_SCHEMA}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )

    saved_count = spark.table(full_name).count()

    print(f"Saved: {full_name}")
    print(f"Rows: {saved_count}")

    return saved_count
          
    

In [0]:
silver_tables = {
    "silver_store_sales": silver_sales,
    "silver_customers": silver_customers,
    "silver_items": silver_items,
    "silver_stores": silver_stores,
    "silver_dates": silver_dates,
    "silver_sales_rejected": all_rejected_sales
}

for table_name , df in silver_tables.items():
    save_silver_table(df, table_name)

print("Silver layer creation completed!")

In [0]:
%sql
USE CATALOG `workspace`;
USE SCHEMA retail_sales;

SELECT
    COUNT(*) AS total_sales_lines,
    ROUND(SUM(net_revenue), 2) AS total_revenue,
    SUM(quantity) AS total_quantity
FROM silver_store_sales;

In [0]:
bronze_count = bronze_sales.count()
silver_count = silver_sales.count()
rejected_count = all_rejected_sales.count()

exact_duplicates_removed = (
    valid_sales.count()
    - sales_no_exact_duplicates.count()
)

quality_report = [
    ("bronze_rows", bronze_count),
    ("silver_rows", silver_count),
    ("rejected_rows", rejected_count),
    ("exact_duplicated_removed", exact_duplicates_removed),
    ("unmatched_item_rows", unmatched_items),
    ("unmatched_customer_rows", unmatched_customers),
    ("unmatched_store_rows", unmatched_stores),
    ("unmatched_date_rows", unmatched_dates)
]

quality_df = spark.createDataFrame(
    quality_report,
    ["metric", "value"]
)

display(quality_df)



In [0]:
assert bronze_count == (
    silver_count
    + rejected_count
    + exact_duplicates_removed
), "Row reconciliation failed!"

assert silver_sales.filter(
    F.col("item_key").isNull() |
    F.col("ticket_number").isNull() |
    F.col("quantity").isNull() |
    (F.col("quantity") <= 0) |
    F.col("net_revenue").isNull() |
    (F.col("net_revenue") < 0)
).count() == 0, "Invalid sales records found!"

remaining_duplicate_keys = (
    silver_sales
    .groupBy("ticket_number", "item_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert remaining_duplicate_keys == 0, (
    "Duplicate transaction keys remain!"
)

print("All Silver sales quality check passed!")